# RAG Colab Notebook
This single notebook runs in Google Colab. It installs dependencies and provides an interactive `ipywidgets` UI to upload a PDF, choose a domain (Media, Law, Telecom, General), and either ask questions or generate a structured summary using a Retrieval-Augmented Generation (RAG) pipeline.

Run the first code cell to install dependencies and set your `OPENAI_API_KEY` value (in Cell 1). Then run the second cell to show the UI.

In [ ]:
# Colab Cell 1 — Install dependencies
# Run this cell first, then RESTART THE RUNTIME (Runtime → Restart runtime),
# then run Cell 2 to launch the UI.

!pip install -q \
  openai \
  langchain-text-splitters \
  pypdf \
  ipywidgets

# Enable ipywidgets in Colab
try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass

# Quick version check
import importlib.metadata as _m
for _p in ["openai", "langchain-text-splitters", "pypdf", "ipywidgets"]:
    try:    print(f"  {_p}=={_m.version(_p)}")
    except: print(f"  {_p} not found")

print("\nDone. Restart runtime now, then run Cell 2.")


In [ ]:
# Colab Cell 2 — RAG UI
# Run after Cell 1 (restart runtime first if Cell 1 just finished).

import io, re, html as _html, threading
import numpy as np
from pypdf import PdfReader
from IPython.display import display
import ipywidgets as widgets

try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass

# ── Pure-Python vector store (no chromadb → no cross-thread DB issues) ────────
# Chunks and their embeddings live in plain Python lists, shared across threads.
# Cosine similarity is done with numpy — entirely thread-safe.

import openai as _oai
from langchain_text_splitters import RecursiveCharacterTextSplitter

_state = {
    "chunks":      [],   # list[str]
    "embeddings":  [],   # list[list[float]]  (1536-dim for text-embedding-3-small)
    "indexed_files": [],
}

# ── OpenAI helpers — synchronous, safe to call from any thread ────────────────

def _key():
    return api_key_input.value.strip()

def _embed(texts: list[str]) -> list[list[float]]:
    resp = _oai.OpenAI(api_key=_key()).embeddings.create(
        model="text-embedding-3-small", input=texts
    )
    return [item.embedding for item in resp.data]

def _chat(prompt: str) -> str:
    resp = _oai.OpenAI(api_key=_key()).chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return resp.choices[0].message.content

# ── PDF extraction ────────────────────────────────────────────────────────────

def _pdf_to_text(b: bytes) -> str:
    reader = PdfReader(io.BytesIO(b))
    pages = []
    for p in reader.pages:
        try:
            pages.append(p.extract_text() or "")
        except Exception:
            pages.append("")
    return "\n\n".join(pages)

_SPLITTER = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# ── Indexing ──────────────────────────────────────────────────────────────────

def index_files(files: list[tuple[str, bytes]]):
    """Embed PDF files and append to shared in-memory store."""
    for fname, pdf_bytes in files:
        _set_status(f"Extracting '{fname}'…", "#aaa")
        text   = _pdf_to_text(pdf_bytes)
        chunks = _SPLITTER.split_text(text)
        if not chunks:
            _set_status(f"No extractable text in '{fname}' — skipped.", "orange")
            continue
        batch_size = 100
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i : i + batch_size]
            _set_status(
                f"Embedding {i+1}–{i+len(batch)}/{len(chunks)} chunks for '{fname}'…", "#aaa"
            )
            embs = _embed(batch)
            _state["chunks"].extend(batch)
            _state["embeddings"].extend(embs)
        if fname not in _state["indexed_files"]:
            _state["indexed_files"].append(fname)
        _refresh_indexed()

# ── Retrieval — numpy cosine similarity ───────────────────────────────────────

def _retrieve(query: str, n: int = 5) -> list[str]:
    if not _state["embeddings"]:
        return []
    q_emb  = np.array(_embed([query])[0], dtype=np.float32)
    matrix = np.array(_state["embeddings"], dtype=np.float32)
    # normalised dot-product = cosine similarity
    norms  = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-10
    sims   = (matrix / norms) @ (q_emb / (np.linalg.norm(q_emb) + 1e-10))
    top_n  = int(min(n, len(sims)))
    idxs   = np.argsort(sims)[-top_n:][::-1]
    return [_state["chunks"][i] for i in idxs]

# ── Prompts ───────────────────────────────────────────────────────────────────

_NOT_FOUND = "I cannot find that information in the provided document."

_QA_TMPL = (
    "You are a helpful assistant specialising in {domain}.\n"
    "Answer ONLY using the CONTEXT below.\n"
    "If the answer is not there, reply EXACTLY: \"{not_found}\"\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n\nAnswer:"
)
_SUMMARY_TMPL = (
    "You are a summarisation assistant specialising in {domain}.\n"
    "Produce a structured summary with clear headings from the CONTEXT below.\n\n"
    "CONTEXT:\n{context}"
)

# ── RAG pipeline — no async, no langchain chains ──────────────────────────────

def query_with_rag(question: str, domain: str) -> str:
    chunks  = _retrieve(question)
    context = "\n\n".join(chunks) if chunks else ""
    answer  = _chat(
        _QA_TMPL.format(domain=domain, context=context,
                        question=question, not_found=_NOT_FOUND)
    )
    if _NOT_FOUND not in answer:
        return answer
    # Fallback to general knowledge
    fb = _chat(
        f"Question: {question}\n\n"
        "The uploaded document didn't contain this information. "
        "Please answer from your general knowledge."
    )
    return "⚠️ **Not found in document** — answering from general knowledge:\n\n" + fb

def generate_summary(domain: str) -> str:
    chunks  = _retrieve("summary overview key points introduction conclusion", n=8)
    context = "\n\n".join(chunks) if chunks else ""
    return _chat(_SUMMARY_TMPL.format(domain=domain, context=context))

# ── Markdown → HTML ───────────────────────────────────────────────────────────

def _md(text: str) -> str:
    out = []
    for line in text.split("\n"):
        if line.startswith("### "):
            out.append(f"<h4 style='margin:6px 0 3px'>{_html.escape(line[4:])}</h4>")
        elif line.startswith("## "):
            out.append(f"<h3 style='margin:8px 0 3px'>{_html.escape(line[3:])}</h3>")
        elif line.startswith("# "):
            out.append(f"<h2 style='margin:10px 0 3px'>{_html.escape(line[2:])}</h2>")
        elif line.startswith(("- ", "* ")):
            out.append(f"<li style='margin:2px 0'>{_html.escape(line[2:])}</li>")
        elif not line.strip():
            out.append("<br>")
        else:
            s = _html.escape(line)
            s = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", s)
            s = re.sub(r"\*(.+?)\*",     r"<i>\1</i>", s)
            out.append(f"<p style='margin:3px 0'>{s}</p>")
    return "".join(out)

# ── Widgets ───────────────────────────────────────────────────────────────────

# — API key row —
api_key_input  = widgets.Password(placeholder="sk-…", description="OpenAI Key:",
                                   layout=widgets.Layout(width="380px"))
api_key_status = widgets.HTML('<span style="color:orange">Enter your key above</span>')

def _on_key(change):
    k = change["new"].strip()
    api_key_status.value = (
        '<span style="color:lightgreen">&#10003; Key ready</span>'
        if k.startswith("sk-") and len(k) > 20
        else '<span style="color:orange">Needs a valid sk-… key</span>'
    )
api_key_input.observe(_on_key, names="value")

# — Domain row —
_DEFAULTS      = ["Media", "Law", "Telecom", "General"]
domain_dd      = widgets.Dropdown(options=_DEFAULTS, value="General",
                                   description="Category:", layout=widgets.Layout(width="220px"))
new_cat_txt    = widgets.Text(placeholder="e.g. Finance…",
                               layout=widgets.Layout(width="200px"))
add_cat_btn    = widgets.Button(description="+ Add",    button_style="warning", layout=widgets.Layout(width="72px"))
rm_cat_btn     = widgets.Button(description="✕ Remove", button_style="danger",  layout=widgets.Layout(width="90px"))
cat_msg        = widgets.HTML("")

def _add_cat(b):
    nm = new_cat_txt.value.strip()
    if not nm:           cat_msg.value = '<span style="color:orange">Type a name.</span>'; return
    opts = list(domain_dd.options)
    if nm in opts:       cat_msg.value = f'<span style="color:orange">"{nm}" exists.</span>'; return
    opts.append(nm); domain_dd.options = opts; domain_dd.value = nm
    new_cat_txt.value = ""
    cat_msg.value = f'<span style="color:lightgreen">Added "{nm}"</span>'
add_cat_btn.on_click(_add_cat)

def _rm_cat(b):
    cur = domain_dd.value
    if cur in _DEFAULTS: cat_msg.value = f'<span style="color:orange">Cannot remove built-in "{cur}".</span>'; return
    opts = [o for o in domain_dd.options if o != cur]
    domain_dd.options = opts; domain_dd.value = opts[-1] if opts else None
    cat_msg.value = f'<span style="color:lightgreen">Removed "{cur}"</span>'
rm_cat_btn.on_click(_rm_cat)

# — Upload row —
upload_out  = widgets.Output(layout=widgets.Layout(border="1px dashed #555", padding="6px",
                              min_height="28px", margin="4px 0"))
upload_btn  = widgets.Button(description="📂 Upload PDF(s)", button_style="info",
                              layout=widgets.Layout(width="160px"))
clear_btn   = widgets.Button(description="Clear all",       button_style="danger",
                              layout=widgets.Layout(width="90px"))

# — Indexed-files display — placed at TOP LEVEL so thread updates always render —
indexed_html = widgets.HTML('<i style="color:#888">No documents indexed yet.</i>',
                             layout=widgets.Layout(margin="4px 0"))

def _refresh_indexed():
    files = _state["indexed_files"]
    if not files:
        indexed_html.value = '<i style="color:#888">No documents indexed yet.</i>'
        return
    rows = "".join(
        f'<li style="color:lightgreen;margin:1px 0">&#10003; {_html.escape(f)}</li>'
        for f in files
    )
    n = len(files)
    indexed_html.value = (
        f'<b style="color:#ccc">Indexed ({n} doc{"s" if n!=1 else ""}):</b> '
        f'<ul style="margin:2px 0;padding-left:16px">{rows}</ul>'
    )

# — Status + Answer —
status_html = widgets.HTML('<i style="color:#888">Ready.</i>')
answer_html = widgets.HTML(
    '<div style="color:#888;font-style:italic;padding:8px">No answer yet.</div>'
)

def _set_status(msg, color="#ccc"):
    status_html.value = f'<span style="color:{color}">{_html.escape(str(msg))}</span>'

def _set_answer(text):
    answer_html.value = (
        '<div style="font-family:sans-serif;font-size:14px;line-height:1.7;'
        'padding:10px;border:1px solid #444;border-radius:4px;margin-top:4px">'
        + _md(text) + '</div>'
    )

# — Upload handler —
def on_upload(b):
    if not _key(): _set_status("Enter your OpenAI API key first.", "orange"); return
    upload_out.clear_output()
    try:
        from google.colab import files as _gf
    except ImportError:
        with upload_out: print("Not in Colab — cannot use files.upload()."); return

    with upload_out:
        print("Opening file picker — choose one or more PDFs…")
        try:
            raw = _gf.upload()
        except Exception as ex:
            print("Upload error:", ex); return
    upload_out.clear_output()
    if not raw:
        _set_status("No files were uploaded.", "#aaa"); return

    pdfs    = [(fn, bytes(c)) for fn, c in raw.items() if fn.lower().endswith(".pdf")]
    skipped = [fn for fn in raw if not fn.lower().endswith(".pdf")]
    if skipped:
        _set_status(f"Skipped non-PDF: {', '.join(skipped)}", "orange")
    if not pdfs:
        _set_status("No PDF files found — please upload .pdf files.", "orange"); return

    def _worker():
        try:
            index_files(pdfs)
            n = len(_state["indexed_files"])
            _set_status(
                f"✓ {n} doc(s) indexed and ready. Ask a question or generate a summary.",
                "lightgreen",
            )
        except Exception as ex:
            _set_status(f"Indexing failed: {ex}", "tomato")
    threading.Thread(target=_worker, daemon=True).start()
upload_btn.on_click(on_upload)

def on_clear(b):
    _state.update({"chunks": [], "embeddings": [], "indexed_files": []})
    _refresh_indexed()
    answer_html.value = '<div style="color:#888;font-style:italic;padding:8px">No answer yet.</div>'
    _set_status("Cleared. Upload new PDF(s) to start again.", "#aaa")
clear_btn.on_click(on_clear)

# — Ask —
ask_text = widgets.Text(description="Question:", layout=widgets.Layout(width="65%"))
ask_btn  = widgets.Button(description="Ask", button_style="primary",
                           layout=widgets.Layout(width="80px"))

def on_ask(b):
    if not _key():                   _set_status("Enter your OpenAI API key first.", "orange"); return
    if not _state["embeddings"]:     _set_status("No documents indexed. Upload a PDF first.", "orange"); return
    q = ask_text.value.strip()
    if not q:                        _set_status("Type a question first.", "orange"); return
    domain = domain_dd.value
    ask_btn.disabled = True
    n = len(_state["indexed_files"])
    _set_status(f"Searching {n} doc(s)…", "#aaa")
    answer_html.value = '<div style="color:#aaa;padding:8px"><i>Thinking…</i></div>'
    def _worker():
        try:
            ans = query_with_rag(q, domain)
            _set_answer(ans)
            _set_status("Done ✓", "lightgreen")
        except Exception as ex:
            _set_status(f"Query failed: {ex}", "tomato")
            answer_html.value = (
                f'<div style="color:tomato;padding:8px"><b>Error:</b> {_html.escape(str(ex))}</div>'
            )
        finally:
            ask_btn.disabled = False
    threading.Thread(target=_worker, daemon=True).start()
ask_btn.on_click(on_ask)

# — Summary —
summary_btn = widgets.Button(description="Generate Summary", button_style="info",
                              layout=widgets.Layout(width="165px"))

def on_summary(b):
    if not _key():               _set_status("Enter your OpenAI API key first.", "orange"); return
    if not _state["embeddings"]: _set_status("No documents indexed. Upload a PDF first.", "orange"); return
    domain = domain_dd.value
    summary_btn.disabled = True
    answer_html.value = '<div style="color:#aaa;padding:8px"><i>Generating summary…</i></div>'
    def _worker():
        try:
            summ = generate_summary(domain)
            _set_answer(summ)
            _set_status("Summary ready ✓", "lightgreen")
        except Exception as ex:
            _set_status(f"Summary failed: {ex}", "tomato")
            answer_html.value = (
                f'<div style="color:tomato;padding:8px"><b>Error:</b> {_html.escape(str(ex))}</div>'
            )
        finally:
            summary_btn.disabled = False
    threading.Thread(target=_worker, daemon=True).start()
summary_btn.on_click(on_summary)

# ── Layout ────────────────────────────────────────────────────────────────────

left_box = widgets.VBox([
    widgets.HTML("<b>Step 2 — Upload Document(s)</b>"),
    widgets.HBox([domain_dd]),
    widgets.HBox([new_cat_txt, add_cat_btn, rm_cat_btn]),
    cat_msg,
    widgets.HBox([upload_btn, clear_btn]),
    upload_out,
])
right_box = widgets.VBox([
    widgets.HTML("<b>Step 3 — Ask or Summarise</b>"),
    widgets.HTML('<span style="font-size:12px;color:#aaa">Upload a doc first, then ask unlimited questions.</span>'),
    widgets.HBox([ask_text, ask_btn]),
    widgets.HTML('<span style="font-size:12px;color:#aaa;margin:4px 0;display:block">— or —</span>'),
    summary_btn,
])

ui = widgets.VBox([
    widgets.HTML("<b>Step 1 — OpenAI API Key</b>"),
    widgets.HBox([api_key_input, api_key_status]),
    widgets.HTML("<hr style='margin:8px 0'>"),
    widgets.HBox([left_box, right_box], layout=widgets.Layout(gap="24px")),
    widgets.HTML("<hr style='margin:8px 0'>"),
    # indexed_html lives here — TOP LEVEL, not inside HBox — so thread updates render
    indexed_html,
    widgets.HTML("<b style='display:block;margin-top:6px'>Status</b>"),
    status_html,
    widgets.HTML("<b style='display:block;margin-top:8px'>Answer</b>"),
    answer_html,
], layout=widgets.Layout(padding="12px", max_width="960px"))

display(ui)
